# Plot from results

In [6]:
from pathlib import Path
import pandas as pd

def load_benchmark_results(
    c_count,
    q_count,
    t_count,
    base_dir="results"
):
    results_dir = (
        Path(base_dir)
        / f"{c_count}_{q_count}_{t_count}"
    )

    benchmark_path = results_dir / "benchmark_results.csv"

    if not benchmark_path.exists():
        raise FileNotFoundError(
            f"No benchmark results found at {benchmark_path}"
        )

    benchmark_df = pd.read_csv(
        benchmark_path
    )

    print(
        f"Loaded {len(benchmark_df)} rows from {benchmark_path}"
    )

    return benchmark_df

def load_ratio_results(
    c_count,
    q_count,
    t_count,
    base_dir="results"
):
    results_dir = (
        Path(base_dir)
        / f"{c_count}_{q_count}_{t_count}"
    )

    ratio_path = results_dir / "ratio_results.csv"

    if not ratio_path.exists():
        raise FileNotFoundError(
            f"No benchmark results found at {ratio_path}"
        )

    ratio_df = pd.read_csv(
        ratio_path
    )

    print(
        f"Loaded {len(ratio_df)} rows from {ratio_path}"
    )

    return ratio_df

c_count = 49
q_count = 1000
t_count = 5000

# Create directory
output_dir = Path("heat_map") / f"{c_count}_{q_count}_{t_count}"
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_df = load_benchmark_results(c_count, q_count, t_count)
ratio_df = load_ratio_results(c_count, q_count, t_count)

Loaded 98 rows from results/49_1000_5000/benchmark_results.csv
Loaded 49 rows from results/49_1000_5000/ratio_results.csv


# Generalized Heat Map

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def plot_heatmap(
    df,
    value_column,
    title,
    filename=None,
    cmap="coolwarm",
    center=None,
    threshold=None,
    cbar_label=None,
    fmt=".3g"
):

    # Create heatmap from exact circuit points
    heatmap_data = (
        df.groupby(
            ["logical_qubits", "t_count"]
        )[value_column]
        .mean()
        .unstack()
    )

    # Sort axes
    heatmap_data = heatmap_data.sort_index(ascending=True)
    heatmap_data = heatmap_data.sort_index(axis=1, ascending=True)


    # Scale figure based on grid size
    n_cols = heatmap_data.shape[1]
    n_rows = heatmap_data.shape[0]

    cell_size = 0.6

    fig, ax = plt.subplots(
        figsize=(
            max(8, n_cols * cell_size),
            max(6, n_rows * cell_size)
        ),
        dpi=150
    )


    # Optional masking for outliers
    if threshold is not None:
        masked_data = heatmap_data.mask(
            heatmap_data > threshold
        )
        vmin = masked_data.min().min()
        vmax = masked_data.max().max()
    else:
        vmin = heatmap_data.min().min()
        vmax = heatmap_data.max().max()



    sns.heatmap(
        heatmap_data,
        cmap=cmap,
        center=center,
        vmin=vmin,
        vmax=vmax,
        annot=True,
        fmt=fmt,
        annot_kws={
            "fontsize": 10
        },
        linewidths=0,
        square=True,
        cbar_kws={
            "label": cbar_label if cbar_label else value_column,
            "shrink": 0.8
        },
        ax=ax
    )


    # Black out masked cells
    if threshold is not None:
        mask = heatmap_data > threshold

        for y, row in enumerate(mask.values):
            for x, value in enumerate(row):
                if value:
                    ax.add_patch(
                        plt.Rectangle(
                            (x, y),
                            1,
                            1,
                            fill=True,
                            color="black"
                        )
                    )


    # Put smallest values bottom-left
    ax.invert_yaxis()


    ax.set_xlabel(
        "T Count"
    )

    ax.set_ylabel(
        "Logical Qubits"
    )

    ax.set_title(
        title
    )


    ax.set_xticklabels(
        heatmap_data.columns,
        rotation=45,
        ha="right"
    )

    ax.set_yticklabels(
        heatmap_data.index,
        rotation=0
    )


    plt.tight_layout()


    if filename:
        plt.savefig(
            filename,
            bbox_inches="tight",
            dpi=150
        )
        plt.close()
    else:
        plt.show()

    return fig

# Plot Ratios

In [8]:
from pathlib import Path

# Create directory
output_dir = Path("heat_map") / f"{c_count}_{q_count}_{t_count}"
output_dir.mkdir(parents=True, exist_ok=True)

# File paths
total_path = output_dir / "total_qubit_ratio_heatmap.png"
compute_path = output_dir / "compute_qubit_ratio_heatmap.png"
factory_path = output_dir / "factory_qubit_ratio_heatmap.png"

fig1 = plot_heatmap(
    ratio_df,
    "total_ratio",
    "Total Physical Qubit Ratio\nQualtran / Azure",
    total_path,
    cmap="coolwarm",
    # center=1,
    threshold=3.0,
    cbar_label="Qualtran / Azure"
)

fig2 = plot_heatmap(
    ratio_df,
    "compute_ratio",
    "Compute Qubit Ratio\nQualtran / Azure",
    compute_path,
    cmap="coolwarm",
    # center=1,
    threshold=3.0,
    cbar_label="Qualtran / Azure"
)

fig3 = plot_heatmap(
    ratio_df,
    "factory_ratio",
    "Factory Qubit Ratio\nQualtran / Azure",
    factory_path,
    cmap="coolwarm",
    # center=1,
    threshold=3.0,
    cbar_label="Qualtran / Azure"
)

# Plot Other Results

In [9]:
azure_df = benchmark_df[
    benchmark_df["estimator_family"] == "Azure"
].copy()

qualtran_df = benchmark_df[
    benchmark_df["estimator_family"]=="Qualtran"
].copy()


factories_ratio_df = pd.merge(
    azure_df,
    qualtran_df,
    on=[
        "circuit_name",
        "t_count",
        "logical_qubits",
    ],
    suffixes=("_azure", "_qualtran")
)

factories_ratio_df["total_ratio"] = (
    factories_ratio_df["num_factories_qualtran"]
    /
    factories_ratio_df["num_factories_azure"]
)

fig = plot_heatmap(
    factories_ratio_df,
    "total_ratio",
    "Factory Ratio",
    output_dir / "num_factories_ratio.png",
    cmap="coolwarm",
    cbar_label="Qualtran / Azure",
)

# Number of factories when factories is the same
fig = plot_heatmap(
    azure_df,
    "num_factories",
    "Number of Magic State Factories",
    output_dir / "num_factories_heatmap_azure.png",
    cmap="viridis",
    # center=1,
    cbar_label="Number of Factories"
)

fig = plot_heatmap(
    qualtran_df,
    "num_factories",
    "Number of Magic State Factories",
    output_dir / "num_factories_heatmap_qualtran.png",
    cmap="viridis",
    # center=1,
    cbar_label="Number of Factories"
)

# Code Distance
fig = plot_heatmap(
    azure_df,
    "code_distance",
    "Code Distance",
    output_dir / "code_distance_heatmap_azure.png",
    cmap="viridis",
    # center=1,
    cbar_label="Code Distance"
)

fig = plot_heatmap(
    qualtran_df,
    "code_distance",
    "Code Distance",
    output_dir / "code_distance_heatmap_qualtran.png",
    cmap="viridis",
    # center=1,
    cbar_label="Code Distance"
)

# Factory fraction
azure_df["factory_fraction"] = (
    azure_df["factory_qubits"]
    /
    azure_df["total_qubits"]
)

fig = plot_heatmap(
    azure_df,
    "factory_fraction",
    "Factory Qubit Fraction",
    output_dir / "factory_fraction_azure.png",
    cmap="flare",
    # center=1,
    cbar_label="Factory Qubits / Total Qubits",
    fmt=".3f"
)

qualtran_df["factory_fraction"] = (
    qualtran_df["factory_qubits"]
    /
    qualtran_df["total_qubits"]
)

fig = plot_heatmap(
    qualtran_df,
    "factory_fraction",
    "Factory Qubit Fraction",
    output_dir / "factory_fraction_qualtran.png",
    cmap="flare",
    # center=1,
    cbar_label="Factory Qubits / Total Qubits",
    fmt=".3f"
)

# Logical error rate + ratio
fig = plot_heatmap(
    azure_df,
    "logical_error_rate",
    "Logical Error Rate",
    output_dir / "logical_error_rate_azure.png",
    cmap="crest",
    # center=1,
    cbar_label="Azure",
    fmt=".3f"
)

fig = plot_heatmap(
    qualtran_df,
    "logical_error_rate",
    "Logical Error Rate",
    output_dir / "logical_error_rate_qualtran.png",
    cmap="crest",
    # center=1,
    cbar_label="Qualtran",
    fmt=".3f"
)

logical_error_ratio_df = pd.merge(
    azure_df,
    qualtran_df,
    on=[
        "circuit_name",
        "t_count",
        "logical_qubits",
    ],
    suffixes=("_azure", "_qualtran")
)

logical_error_ratio_df["total_ratio"] = (
    logical_error_ratio_df["logical_error_rate_qualtran"]
    /
    logical_error_ratio_df["logical_error_rate_azure"]
)

fig = plot_heatmap(
    logical_error_ratio_df,
    "total_ratio",
    "Logical Error Ratio",
    output_dir / "logical_error_ratio.png",
    cmap="crest",
    # center=1,
    cbar_label="Qualtran / Azure",
)

In [10]:
# Runtime + ratio
fig = plot_heatmap(
    azure_df,
    "runtime_seconds",
    "Runtime",
    output_dir / "runtime_azure.png",
    cmap="coolwarm",
    cbar_label="Azure",
)

fig = plot_heatmap(
    qualtran_df,
    "runtime_seconds",
    "Runtime",
    output_dir / "runtime_qualtran.png",
    cmap="coolwarm",
    cbar_label="Qualtran",
)

runtime_ratio_df = pd.merge(
    azure_df,
    qualtran_df,
    on=[
        "circuit_name",
        "t_count",
        "logical_qubits",
    ],
    suffixes=("_azure", "_qualtran")
)

runtime_ratio_df["total_ratio"] = (
    runtime_ratio_df["runtime_seconds_qualtran"]
    /
    runtime_ratio_df["runtime_seconds_azure"]
)

fig = plot_heatmap(
    runtime_ratio_df,
    "total_ratio",
    "Runtime Ratio",
    output_dir / "runtime_ratio.png",
    cmap="coolwarm",
    cbar_label="Qualtran / Azure",
)